# Phase Fitting + Structure-Agnostic Peak Fitting (Single Pattern)

Two complementary fitting modes in `ssrl_xrd_tools.analysis.fitting`:

| Mode                    | Function                | When you'd use it                             |
| ----------------------- | ----------------------- | --------------------------------------------- |
| **Structure-informed**  | `PhaseFitter`           | You know what phases are present (CIF available); want phase fractions / lattice parameters |
| **Structure-agnostic**  | `fit_peaks`             | Identify / fit individual peaks without prior structural knowledge                          |

Both share the same lmfit model zoo and background utilities.  This
notebook fits the **same pattern** with both and compares results.

**Adapt the paths + `q_range` in the configuration cell.**


## Imports

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from ssrl_xrd_tools.analysis.phase import PhaseModel
from ssrl_xrd_tools.analysis.fitting import (
    PhaseFitter,
    fit_peaks,
    extract_peaks,
    snip_1d,
)

## ✏️ Configuration

**Edit the cell below**, then run the rest of the notebook top-to-bottom.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                  EDIT THIS CELL — paths + parameters                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── REQUIRED ───────────────────────────────────────────────────────────
data_file = Path('~/data/integrated_pattern.npz').expanduser()  # ← REPLACE
cif_dir   = Path('~/data/cifs').expanduser()                    # ← REPLACE
cif_files = {
    'alpha':  cif_dir / 'phase_alpha.cif',     # ← REPLACE with your phases
    'beta':   cif_dir / 'phase_beta.cif',
}
wavelength_A = 1.5406              # ← REPLACE — Cu Kα default, set to YOUR wavelength

# ── Fitting parameters ─────────────────────────────────────────────────
q_range    = (1.0, 5.0)            # fit window in Å⁻¹
background = 'snip'                # 'snip' | 'chebyshev_5' | 'constant' | ...

### Validate the config

In [ ]:
assert data_file.is_file(), f'data_file not found: {data_file}'
assert cif_dir.is_dir(), f'cif_dir not found: {cif_dir}'
for name, path in cif_files.items():
    assert path.is_file(), f'CIF {name!r} not found: {path}'
print(f'OK — pattern + {len(cif_files)} CIF(s) present.')

## Load a single integrated pattern

We assume the pattern is stored as an `npz` with `q` and `intensity`
arrays.  Adapt this cell if you're reading from NeXus, XYE, etc.

In [ ]:
data = np.load(data_file)
q, intensity = np.asarray(data['q']), np.asarray(data['intensity'])

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(q, intensity, lw=0.8)
ax.set_xlabel('q (Å⁻¹)'); ax.set_ylabel('Intensity')
ax.set_xlim(*q_range)
plt.tight_layout(); plt.show()

## Structure-informed: `PhaseFitter`

Build a `PhaseModel` per phase (peak positions + template intensities
derived from the CIF via pymatgen), feed them into a `PhaseFitter`,
fit, plot.

The q-range filter is applied when adding the phase to the fitter
(``add_phase(phase, q_range=...)``), not in ``calculate_peaks`` — the
phase carries every reflection until you trim it at fit time.

In [ ]:
phases = [
    PhaseModel.from_cif(path, name=name)
    for name, path in cif_files.items()
]
for ph in phases:
    ph.calculate_peaks(wavelength=wavelength_A)
    print(f'{ph.name}: {len(ph.peaks)} reflections in CIF '
          f'(full pattern, before q-window trim)')

In [ ]:
fitter = PhaseFitter(q, intensity, prefit_background=background)
for ph in phases:
    fitter.add_phase(ph, q_range=q_range)
phase_result = fitter.fit(q_range=q_range)
print(f'PhaseFitter redχ² = {phase_result.redchi:.3f}')
print(f'Phase fractions: {phase_result.phase_fractions()}')

In [ ]:
# best_fit lives on the underlying lmfit ModelResult.  We also
# break out the per-phase components via lmfit's eval_components
# (a {component_name: ndarray} mapping).
mask = (q >= q_range[0]) & (q <= q_range[1])
best_fit = phase_result.lmfit_result.best_fit
components = phase_result.lmfit_result.eval_components()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(q[mask], intensity[mask], 'k.', ms=2, label='data')
ax.plot(q[mask], best_fit,        'C3-', lw=1.2, label='PhaseFitter')
for name, comp in components.items():
    if name.startswith('p') and '_' in name:        # p0_, p1_, ...
        ax.plot(q[mask], comp, '--', lw=0.7, label=name)
ax.set_xlabel('q (Å⁻¹)'); ax.set_ylabel('Intensity')
ax.legend(loc='best')
plt.tight_layout(); plt.show()

## Structure-agnostic: `fit_peaks`

`fit_peaks` accepts an explicit list of peak positions (or
auto-estimates them when ``positions=None``) and fits each as an
individual pseudo-Voigt on top of the chosen background.  Useful when
phases aren't known, or as a cross-check.

In [ ]:
# A simple peak picker: SNIP-subtract then take the largest few maxima.
# In production you'd usually pass `positions=` from a manual list or
# from scipy.signal.find_peaks.
from scipy.signal import find_peaks

mask = (q >= q_range[0]) & (q <= q_range[1])
y_bs = intensity[mask] - snip_1d(intensity[mask], w=50)
idx, _ = find_peaks(y_bs, prominence=0.05 * y_bs.max(), distance=5)
positions = q[mask][idx]
print(f'Detected {len(positions)} peaks at q ≈ '
      f'{[f"{p:.3f}" for p in positions]}')

In [ ]:
peak_result = fit_peaks(
    q[mask], intensity[mask],
    positions=positions,
    model='pseudovoigt',
    background=background,
)
print(f'fit_peaks redχ² = {peak_result.fit_result.redchi:.3f}')
print(f'Fit {peak_result.n_peaks} peaks ({peak_result.model_name} on '
      f'{peak_result.background_name} background)')
print('First 3 peaks (centre / sigma / amplitude):')
for c, s, a in list(zip(peak_result.peak_centers,
                        peak_result.peak_sigmas,
                        peak_result.peak_amplitudes))[:3]:
    print(f'  q={c:.4f}  σ={s:.4f}  amp={a:.2f}')

## Compare side-by-side

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
ax, axr = axes
ax.plot(q[mask], intensity[mask], 'k.', ms=2, label='data')
ax.plot(q[mask], best_fit,             'C3-', lw=1.0,
        label=f'PhaseFitter (redχ²={phase_result.redchi:.2f})')
ax.plot(q[mask], peak_result.best_fit, 'C0-', lw=1.0,
        label=f'fit_peaks (redχ²={peak_result.fit_result.redchi:.2f})')
ax.set_ylabel('Intensity'); ax.legend()

axr.plot(q[mask], intensity[mask] - best_fit,             'C3-', lw=0.6,
         label='PhaseFitter residual')
axr.plot(q[mask], intensity[mask] - peak_result.best_fit, 'C0-', lw=0.6,
         label='fit_peaks residual')
axr.axhline(0, color='k', lw=0.5)
axr.set_xlabel('q (Å⁻¹)'); axr.set_ylabel('residual'); axr.legend(fontsize=8)
plt.tight_layout(); plt.show()

---

### When to prefer which

- **`PhaseFitter`** — when phase identification is the goal (relative
  fractions, lattice parameter trends across a sample series).  See
  notebook **04 — Batch phase fitting** for the sequence pattern.
- **`fit_peaks`** — when you care about individual peak positions /
  widths / areas (peak shifts vs. temperature, line broadening
  analysis).
